# Google QUEST: offline inference

Load the completed run and tokenizer from the attached trained-model dataset. Verify recorded artifacts, predict the competition-mounted test rows, and write `submission.csv`. Internet access is disabled. The notebook uses the package's same input checks, target order and checkpoint loader as local inference.

In [ ]:
from pathlib import Path
import hashlib, importlib.metadata, importlib.util, json, os, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"
wheels = list(Path("/kaggle/input").rglob("quest_labeling-*.whl"))
assert len(wheels) == 1, "Attach exactly one QUEST code wheel"
assets = wheels[0].parent
source = json.loads((assets / "source.json").read_text())
assert hashlib.sha256(wheels[0].read_bytes()).hexdigest() == source["wheel_sha256"]
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",
                "--disable-pip-version-check", str(wheels[0])], check=True)
print({"python": sys.version.split()[0], "versions": {name: importlib.metadata.version(name)
       for name in ["torch", "transformers", "numpy", "pandas", "scikit-learn"]},
       "py7zr": importlib.metadata.version("py7zr") if importlib.util.find_spec("py7zr") else None,
       "code": source})


In [ ]:
import torch
from quest.runtime import predict
from quest.data import sha256
checkpoints = list(Path("/kaggle/input").rglob("model.pt"))
assert len(checkpoints) == 1, "Attach exactly one trained QUEST run"
run_dir = checkpoints[0].parent
manifest = json.loads((run_dir / "manifest.json").read_text())
assert manifest["status"] == "completed"
for name, digest in manifest["artifacts"].items():
    path = run_dir / name
    assert path.is_relative_to(run_dir) and ".." not in Path(name).parts
    assert sha256(path) == digest, name
competition_roots = [p.parent for p in Path("/kaggle/input").rglob("test.csv")
                     if (p.parent / "sample_submission.csv").is_file()]
assert len(competition_roots) == 1
receipt = predict(competition_roots[0], run_dir, Path("/kaggle/working/submission.csv"),
                  device="cuda" if torch.cuda.is_available() else "cpu")
print(receipt)
